# f

In [ ]:
!pip install rembg[cpu] 

In [ ]:
!pip install -U transformers diffusers accelerate

In [ ]:
!pip install xformers

In [ ]:
!pip install gradio

In [1]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np
from rembg import remove

class BannerGenerator:
    def __init__(self, device="cuda"):
        self.device = device
        self.create_pipe()

    def create_pipe(self):
        image_encoder = CLIPVisionModelWithProjection.from_pretrained(
            "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
            torch_dtype=torch.float16
        ).to(self.device)

        controlnet = ControlNetModel.from_pretrained(
            "diffusers/controlnet-canny-sdxl-1.0", 
            torch_dtype=torch.float16
        ).to("cuda")

        self.pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            controlnet=controlnet,
            image_encoder=image_encoder,
            torch_dtype=torch.float16,
            variant="fp16"
        ).to("cuda")

        self.pipe.load_ip_adapter(
            "h94/IP-Adapter", 
            subfolder="sdxl_models", 
            weight_name="ip-adapter-plus_sdxl_vit-h.safetensors"
        )

        self.pipe.enable_model_cpu_offload()
        self.pipe.enable_xformers_memory_efficient_attention()
        self.mask_processor = IPAdapterMaskProcessor() 

    def generate_edge_map(self, image : Image.Image):
        image_np = np.array(image)
        edges = cv2.Canny(image_np, 150, 300)
        edges = edges[:, :, None]
        edges = np.concatenate([edges, edges, edges], axis=2)
        canny_image = Image.fromarray(edges)
        return canny_image
    
    def create_mask(self, image : Image.Image):
        alpha = image.split()[3]
        mask = Image.new("L", image.size, 0) 
        white = Image.new("L", image.size, 255) 
        mask = Image.composite(white, mask, alpha)
        return mask

    def generate(self, 
                 input_image: Image.Image, 
                 prompt: str, 
                 negative_prompt: str = "",
                 ip_scale: float = 0.7, 
                 control_scale: float = 0.8,
                 guidance_scale: float = 7.5,
                 steps: int = 30):

        self.pipe.set_ip_adapter_scale(ip_scale)

        only_product_img = remove(input_image)
    
        msk = self.get_mask(only_product_img)
        canny_image = self.generate_edge_map(input_image)
        processed_mask = self.mask_processor.preprocess(
            [msk], 
            height=1024, 
            width=1024
        )
        image = self.pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=canny_image,     
            ip_adapter_image=only_product_img,
            controlnet_conditioning_scale=control_scale,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            cross_attention_kwargs={"ip_adapter_masks": [processed_mask]}
        ).images[0]
        return image


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
from model import BannerGenerator
import gradio as gr


gen = BannerGenerator()

def process_banner(image, prompt, neg_prompt, ip_scale, ctrl_scale, guidance, steps):
    result = gen.generate(
        input_image=image,
        prompt=prompt,
        negative_prompt=neg_prompt,
        ip_scale=ip_scale,
        control_scale=ctrl_scale,
        guidance_scale=guidance,
        steps=int(steps)
    )
    return result

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("Ad banner Generator (SDLX + ControlNet + Ip-Adapter)")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Product Image")
            prompt = gr.Textbox(
                label="Prompt", 
                placeholder="e.g., Professional product photography of a soda can on a marble table..."
            )
            neg_prompt = gr.Textbox(
                label="Negative Prompt", 
                value="low quality, blurry, distorted, yellow, gold"
            )
            
            with gr.Accordion("Advanced Settings", open=False):
                ip_scale = gr.Slider(0, 1.0, value=0.8, label="IP-Adapter Scale")
                ctrl_scale = gr.Slider(0, 1.0, value=0.8, label="ControlNet Scale")
                guidance = gr.Slider(1.0, 15.0, value=7.5, label="Guidance Scale")
                steps = gr.Number(value=30, label="Steps", precision=0)
            
            run_btn = gr.Button("Generate Banner", variant="primary")
            
        with gr.Column():
            output_img = gr.Image(label="Result")

    run_btn.click(
        fn=process_banner,
        inputs=[input_img, prompt, neg_prompt, ip_scale, ctrl_scale, guidance, steps],
        outputs=[output_img]
    )

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np
from rembg import remove


# 1. Загружаем модели (исправленные импорты)
# Размерность 1024 для ViT-H
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
    torch_dtype=torch.float16
).to("cuda")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

# 2. Инициализация пайплайна (без ошибочных имен)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 3. Загрузка IP-Adapter (Plus версия для сохранения синего цвета)
pipe.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter-plus_sdxl_vit-h.safetensors"
)

# ВКЛЮЧАЕМ ОПТИМИЗАЦИЮ ПАМЯТИ (обязательно для 12GB)
pipe.enable_model_cpu_offload()
pipe.enable_xformers_memory_efficient_attention()


 
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
# 6. Подготовка изображений
product_img = load_image(url).convert("RGB").resize((1024, 1024))

# Создаем карту краев (Canny)
image_np = np.array(product_img)
edges = cv2.Canny(image_np, 100, 200)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 7. Настройка весов (Важно!)
# IP-Adapter scale 0.7 заставит модель "слушаться" цвета оригинала
pipe.set_ip_adapter_scale(0.8)

# 8. Генерация
prompt = "Professional product photography of the red can of soda, placed on a smooth matte light-grey surface, soft studio lighting, long elegant shadows, minimalist aesthetic, high-end commercial style, clean background, 8k resolution, sharp focus."
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

only_product_img = remove(product_img)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,     
    ip_adapter_image=only_product_img,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=30,
    guidance_scale=7.5,
    #cross_attention_kwargs={"ip_adapter_masks": [mask]}
).images[0]

# 9. Просмотр и сохранение
image.save("final_blue_banner.png")
#image.show()
display(image)

In [ ]:
def get_mask(img):
    alpha = img.split()[3]
    mask = Image.new("L", img.size, 0) 
    white = Image.new("L", img.size, 255) 
    mask = Image.composite(white, mask, alpha)
    return mask

In [ ]:
from diffusers.image_processor import IPAdapterMaskProcessor
mask_processor = IPAdapterMaskProcessor()
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
# 6. Подготовка изображений
product_img = load_image(url).convert("RGB").resize((1024, 1024))

# Создаем карту краев (Canny)
image_np = np.array(product_img)
edges = cv2.Canny(image_np, 100, 200)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 7. Настройка весов (Важно!)
# IP-Adapter scale 0.7 заставит модель "слушаться" цвета оригинала
pipe.set_ip_adapter_scale(0.8)

# 8. Генерация
prompt = "Professional product photography of the red can of soda, placed on a smooth matte light-grey surface, soft studio lighting, long elegant shadows, minimalist aesthetic, high-end commercial style, clean background, 8k resolution, sharp focus."
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

only_product_img = remove(product_img)
def autocrop_image(img):
    # Открываем и гарантируем наличие альфа-канала
    
    
    # getbbox возвращает кортеж (left, upper, right, lower)
    bbox = img.getbbox()
    
    if bbox:
        # Обрезаем по найденным координатам
        cropped_img = img.crop(bbox)
        #cropped_img.save(output_path)
        #print(f"Обрезка выполнена. Размер: {cropped_img.size}")
        return cropped_img
msk = get_mask(only_product_img)
NOBG = autocrop_image(only_product_img)
processed_mask = mask_processor.preprocess(
    [msk], 
    height=1024, 
    width=1024
)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,     
    ip_adapter_image=NOBG,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=30,
    guidance_scale=7.5,
    cross_attention_kwargs={"ip_adapter_masks": [processed_mask]}
).images[0]

# 9. Просмотр и сохранение
image.save("final_blue_banner.png")
#image.show()
display(image)

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from diffusers.image_processor import IPAdapterMaskProcessor
from PIL import Image

def get_product_description(pil_image):
    # 1. Загружаем модель и процессор (Salesforce/blip-image-captioning-base)
    # Эта модель весит всего ~900 МБ
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base", 
        torch_dtype=torch.float16
    ).to("cuda")

    # 2. Подготовка изображения
    inputs = processor(pil_image, return_tensors="pt").to("cuda", torch.float16)

    # 3. Генерация текста
    out = model.generate(**inputs)
    description = processor.decode(out[0], skip_special_tokens=True)

    # 4. Очистка памяти (важно для 12GB VRAM)
    del model
    del processor
    torch.cuda.empty_cache()
    
    return description

# Использование в твоем пайплайне:
# product_img = load_image(url).convert("RGB").resize((1024, 1024))
product_name = get_product_description(product_img)
print(f"Робот увидел: {product_name}")

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
import numpy as np
import cv2
from PIL import Image

# --- ЭТАП 1: УМНОЕ ОПИСАНИЕ (Чтобы IP-Adapter не терял цвет) ---
def get_auto_prompt(img):
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base", torch_dtype=torch.float16
    ).to("cuda")
    
    inputs = processor(img, return_tensors="pt").to("cuda", torch.float16)
    out = model.generate(**inputs)
    product_desc = processor.decode(out[0], skip_special_tokens=True)
    
    del model, processor
    torch.cuda.empty_cache()
    return product_desc

# Получаем описание товара автоматически (например, "a blue can of soda")
detected_product = get_auto_prompt(product_img)
print(f"Detected: {detected_product}")

# --- ЭТАП 2: ПОДГОТОВКА CANNY (С размытием для чистоты) ---
image_np = np.array(product_img)
gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0) # Убираем шум
edges = cv2.Canny(blurred, 100, 200)        # Более мягкие пороги для деталей
canny_image = Image.fromarray(np.stack([edges]*3, axis=-1))

# --- ЭТАП 3: ГЕНЕРАЦИЯ ---
# 1. Снижаем IP-scale до 0.65, чтобы не "жарить" картинку
pipe.set_ip_adapter_scale(0.5) 

# 2. Формируем динамический промпт
# Мы используем detected_product, чтобы нейросеть ЗНАЛА цвет и тип объекта
prompt = (
    f"Professional advertising shot of {detected_product}, "
    "placed on a tropical beach with turquoise water and palm shadows, "
    "high-end commercial style, cinematic lighting, sharp focus, 8k resolution"
)

negative_prompt = (
    "blurry, deformed, messy, low quality, distorted text, "
    "bad anatomy, extra limbs, grainy, flickery"
)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,           
    #ip_adapter_image=product_img, 
    controlnet_conditioning_scale=0.7, # Чуть снижаем, чтобы края были мягче
    controlnet_guidance_end=0.6,      # Позволяем модели самой дорисовать блики в конце
    num_inference_steps=35,           # Больше шагов = чище результат
    guidance_scale=7.5,               # Выше CFG = лучшее следование тексту
).images[0]

# --- ЭТАП 4: СОХРАНЕНИЕ ---
image.save("final_banner_auto.png")
display(image)

In [ ]:
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
# 6. Подготовка изображений
product_img = load_image(url).convert("RGB").resize((1024, 1024))

# Создаем карту краев (Canny)
image_np = np.array(product_img)
edges = cv2.Canny(image_np, 150, 300)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 7. Настройка весов (Важно!)
# IP-Adapter scale 0.7 заставит модель "слушаться" цвета оригинала
pipe.set_ip_adapter_scale(0.5)

# 8. Генерация
prompt = "A red soda can on a stylized tropical beach, Studio Ghibli inspired, animated movie style, 2d digital art, cel-shaded, vibrant colors, clean lines, no gradients, sparkling turquoise ocean, fluffy clouds, sunny day."
negative_prompt = "blurry background, gold, yellow, orange, deformed, messy, blurry, low quality"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,           # Держит форму
    ip_adapter_image=product_img, # Держит цвет (синий)
    controlnet_conditioning_scale=1.,
    num_inference_steps=30,
    #controlnet_guidance_end=0.8,
    guidance_scale=5.5,
).images[0]

# 9. Просмотр и сохранение
image.save("final_blue_banner.png")
#image.show()
display(image)

In [ ]:
from IPython.display import display
from diffusers import DPMSolverMultistepScheduler

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, 
    use_karras_sigmas=True
)

url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
# 6. Подготовка изображений
product_img = load_image(url).convert("RGB").resize((1024, 1024))

from transformers import pipeline

# Initialize the depth estimator (runs on GPU)
depth_estimator = pipeline("depth-estimation", model="intel/dpt-hybrid-midas")

def get_depth_map(image):
    # Get depth map
    depth_map = depth_estimator(image)['depth']
    depth_map = np.array(depth_map)
    
    # Standardize format for ControlNet
    depth_map = depth_map[:, :, None]
    depth_map = np.concatenate([depth_map, depth_map, depth_map], axis=2)
    return Image.fromarray(depth_map)

# Prepare your depth map
depth_image = get_depth_map(product_img)

# Создаем карту краев (Canny)
image_np = np.array(product_img)
edges = cv2.Canny(image_np, 150, 300)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 7. Настройка весов (Важно!)
# IP-Adapter scale 0.7 заставит модель "слушаться" цвета оригинала
pipe.set_ip_adapter_scale(0.35)

# 8. Генерация
prompt = "Professional food photography of a classic red soda can, covered in hyper-realistic water droplets and condensation, sitting on a bed of crushed glistening ice, dramatic studio lighting, deep red and white color palette, cinematic atmosphere, 8k, sharp focus, macro details, high contrast, vibrant colors."
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,           # Держит форму
    #ip_adapter_image=product_img, # Держит цвет (синий)
    controlnet_conditioning_scale=0.5,
    num_inference_steps=30,
    guidance_scale=7.5,
).images[0]

# 9. Просмотр и сохранение
image.save("final_blue_banner.png")
display(image)

# Assone

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from PIL import Image
import cv2
from IPython.display import display

import numpy as np

# 1. Load ControlNet (Canny) - The "Structural Guard"
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

# 2. Load SDXL Pipeline
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 4. Memory Optimizations for 12GB VRAM
pipe.enable_model_cpu_offload() 
#pipe.enable_xformers_memory_efficient_attention()
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"

# 5. Process the Product Image
product_img = load_image(url).resize((1024, 1024))

# Generate Canny Edges
image_np = np.array(product_img)
# Adjust 100, 200 to change how many details of the product are kept
edges = cv2.Canny(image_np, 150, 300) 
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 6. Define the "Style" via Prompt
# Use keywords your LoRA was trained on (e.g., 'mrt_style')
prompt = "Professional food photography of a classic red soda can, covered in hyper-realistic water droplets and condensation, sitting on a bed of crushed glistening ice, dramatic studio lighting, deep red and white color palette, cinematic atmosphere, 8k, sharp focus, macro details, high contrast, vibrant colors."
negative_prompt = "text, watermark, blurry, low quality, distorted product, messy background, colorful noise"

# 7. Generate
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,
    controlnet_conditioning_scale=0.5, # 0.7-0.8 is the sweet spot for product ads
    num_inference_steps=35,
    guidance_scale=7,
).images[0]

image.save("generated_picture.png")
display(image)

# conrballs


In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from diffusers.utils import load_image

pipeline = AutoPipelineForText2Image.from_pretrained(
  "stabilityai/stable-diffusion-xl-base-1.0",
  torch_dtype=torch.float16
).to("cuda")
pipeline.load_ip_adapter(
  "h94/IP-Adapter",
  subfolder="sdxl_models",
  weight_name="ip-adapter_sdxl.bin"
)
pipeline.set_ip_adapter_scale(0.8)

In [ ]:
image = load_image("https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg")
pipeline(
    prompt="a polar bear sitting in a chair drinking a milkshake",
    ip_adapter_image=image,
    negative_prompt="deformed, ugly, wrong proportion, low res, bad anatomy, worst quality, low quality",
).images[0]

In [ ]:
from rembg import remove
from PIL import Image
from diffusers.utils import load_image


input_img = load_image("https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg")
output_img = remove(input_img)

output_img.save('mask_result.png')

Base SDLX

In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from diffusers.utils import load_image

pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"

init_image = load_image(url).resize((1024, 1024))

strength_values = [0.4, 0.55, 0.7, 0.85]
guidance_values = [7.0, 10.0, 14.0]

def generate_image(init_image, strength, guidance_scale):
    image = pipe(
        prompt="a luxury commercial photo of a red soda bottle on a black glossy table",
        image=init_image,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=40
    ).images[0]
    return image

grid_results = []
for g_val in guidance_values:
    row_results = []
    for s_val in strength_values:
        
        img = generate_image(init_image, s_val, g_val)
        row_results.append(img)
    grid_results.append(row_results)



In [ ]:
!apt-get update -y
!apt-get install -y fonts-cmu

In [ ]:
from PIL import Image, ImageDraw, ImageFont
num_rows = len(guidance_values)
num_cols = len(strength_values)
img_w, img_h = 1024, 1024 # Размер одного изображения
label_space_x = 450 # Место для подписей Guidance (слева)
label_space_y = 200 # Место для подписей Strength (сверху)

# Создаем большой пустой холст
# Ширина: (Ширина изображений + место слева)
# Высота: (Высота изображений + место сверху)
grid_w = num_cols * img_w + label_space_x
grid_h = num_rows * img_h + label_space_y
collage_image = Image.new('RGB', (grid_w, grid_h), color=(255, 255, 255)) # Белый фон

# Подготовка к рисованию подписей
draw = ImageDraw.Draw(collage_image)
font_size = 70 

try:
    font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    font = ImageFont.truetype(font_path, font_size)
except:
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", font_size)
    except:
        print("Шрифт не найден, используем дефолтный (будет мелко)")
        font = ImageFont.load_default()

# 5. Размещаем изображения и добавляем подписи
for i, row in enumerate(grid_results):
    for j, img in enumerate(row):
        # Вычисляем позицию для вставки изображения
        x_pos = j * img_w + label_space_x
        y_pos = i * img_h + label_space_y
        collage_image.paste(img, (x_pos, y_pos))

        # Добавляем подписи по оси X (Strength) - сверху
        if i == 0:
            str_text = f"Strength: {strength_values[j]}"
            # (Текст, Центрирование по X, Отступ сверху)
            draw.text((x_pos + img_w // 2, 30), str_text, fill=(0, 0, 0), font=font, anchor="mm")

    # Добавляем подписи по оси Y (Guidance) - слева
    gui_text = f"Guidance:\n{guidance_values[i]}"
    # (Текст, Отступ слева, Центрирование по Y)
    draw.text((30, y_pos + img_h // 2), gui_text, fill=(0, 0, 0), font=font, anchor="lm")

# 6. Сохраняем финальный коллаж
#collage_image.save("hyperparameter_study_grid.png")
collage_image.save("output_grid.png")
collage_image.show() # Показать результат
print("Grid saved as output_grid.png")

ШИЗА CONTROLNET AHHH

In [ ]:
def create_triple_collage(img, img1, img2, output_path, padding=20):
    w, h = img.size
    w1, h1 = img1.size
    w2, h2 = img2.size

    collage_width = w + w1 + w2 + (2 * padding)
    collage_height = max(h, h1, h2)

    collage = Image.new('RGB', (collage_width, collage_height), color=(255, 255, 255))

    collage.paste(img, (0, 0))
    collage.paste(img1, (w + padding, 0))

    collage.paste(img2, (w + padding + w1 + padding, 0))

    collage.save(output_path)
    print(f"Коллаж сохранен как {output_path}")
    return collage
def stack_collages_vertically(row1, row2, row3, output_path, padding_v=40):
    width = max(row1.width, row2.width, row3.width)
    total_height = row1.height + row2.height + row3.height + (2 * padding_v)

    final_stack = Image.new('RGB', (width, total_height), color=(255, 255, 255))

    current_y = 0
    
    final_stack.paste(row1, (0, current_y))
    current_y += row1.height + padding_v
    
    final_stack.paste(row2, (0, current_y))
    current_y += row2.height + padding_v
    
    final_stack.paste(row3, (0, current_y))

    final_stack.save(output_path)
    print(f"Вертикальный стек сохранен: {output_path}")
    return final_stack

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np

image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
    torch_dtype=torch.float16
).to("cuda")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")


# ВКЛЮЧАЕМ ОПТИМИЗАЦИЮ ПАМЯТИ (обязательно для 12GB)
pipe.enable_model_cpu_offload()
pipe.enable_xformers_memory_efficient_attention()


 


In [ ]:
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
def gen_collat(url, prompt):

    product_img = load_image(url).convert("RGB").resize((1024, 1024))

    image_np = np.array(product_img)
    edges = cv2.Canny(image_np, 100, 200)
    edges = edges[:, :, None]
    edges = np.concatenate([edges, edges, edges], axis=2)
    canny_image = Image.fromarray(edges)
    prompt = "Professional product photography of the red can of soda, placed on a smooth matte light-grey surface, soft studio lighting, 8k resolution, sharp focus."
    negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=canny_image,     
        controlnet_conditioning_scale=0.8,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]

    image.save("final_blue_banner.png")
    res = create_triple_collage(product_img, canny_image, image, "collage.png")
    return res


In [ ]:
url1 = "https://www.hairbeautyink.com.au/cdn/shop/products/blonde-studio-post-lightening-shampoo-500ml-941238.jpg"
url2 = "https://extrabutterny.com/cdn/shop/files/MSRCXST-1.jpg?v=1688674811&width=1200"
prompt = "Professional commercial photography of a product placed on fine white beach sand, turquoise ocean waves in the blurred background, golden hour sunlight creating long soft shadows, cinematic lighting, water droplets on the surface, 8k resolution, highly detailed, sharp focus, f/1.8."

res = gen_collat(url, prompt)
res1 = gen_collat(url1, prompt)
res2 = gen_collat(url2, prompt)



In [ ]:
#st = stack_collages_vertically(res, res1, res2, "stack.png")

display(gen_collat("https://i.redd.it/xp05tia6bwg21.jpg", prompt))

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np
from rembg import remove


# 1. Загружаем модели (исправленные импорты)
# Размерность 1024 для ViT-H
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
    torch_dtype=torch.float16
).to("cuda")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

# 2. Инициализация пайплайна (без ошибочных имен)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 3. Загрузка IP-Adapter (Plus версия для сохранения синего цвета)
pipe.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter-plus_sdxl_vit-h.safetensors"
)

# ВКЛЮЧАЕМ ОПТИМИЗАЦИЮ ПАМЯТИ (обязательно для 12GB)
pipe.enable_model_cpu_offload()
pipe.enable_xformers_memory_efficient_attention()


 
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
# 6. Подготовка изображений
product_img = load_image(url).convert("RGB").resize((1024, 1024))

# Создаем карту краев (Canny)
image_np = np.array(product_img)
edges = cv2.Canny(image_np, 100, 200)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 7. Настройка весов (Важно!)
# IP-Adapter scale 0.7 заставит модель "слушаться" цвета оригинала
pipe.set_ip_adapter_scale(0.8)

# 8. Генерация
prompt = "Professional product photography of the red can of soda, placed on a smooth matte light-grey surface, soft studio lighting, long elegant shadows, minimalist aesthetic, high-end commercial style, clean background, 8k resolution, sharp focus."
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

only_product_img = remove(product_img)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,     
    ip_adapter_image=only_product_img,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=30,
    guidance_scale=7.5,
    #cross_attention_kwargs={"ip_adapter_masks": [mask]}
).images[0]

# 9. Просмотр и сохранение
image.save("final_blue_banner.png")
#image.show()
display(image)

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.image_processor import IPAdapterMaskProcessor
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np
from rembg import remove
from diffusers.image_processor import IPAdapterMaskProcessor


def get_mask(img):
    alpha = img.split()[3]
    mask = Image.new("L", img.size, 0) 
    white = Image.new("L", img.size, 255) 
    mask = Image.composite(white, mask, alpha)
    return mask


# 1. Загружаем модели (исправленные импорты)
# Размерность 1024 для ViT-H
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
    torch_dtype=torch.float16
).to("cuda")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

# 2. Инициализация пайплайна (без ошибочных имен)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 3. Загрузка IP-Adapter (Plus версия для сохранения синего цвета)
pipe.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter-plus_sdxl_vit-h.safetensors"
)

pipe.enable_model_cpu_offload()
pipe.enable_xformers_memory_efficient_attention()


 
mask_processor = IPAdapterMaskProcessor()
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
product_img = load_image(url).convert("RGB").resize((1024, 1024))

image_np = np.array(product_img)
edges = cv2.Canny(image_np, 150, 300)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

pipe.set_ip_adapter_scale(0.8)

prompt = "Professional product photography of the red can of soda, placed on a smooth matte light-grey surface, soft studio lighting, long elegant shadows, minimalist aesthetic, high-end commercial style, clean background, 8k resolution, sharp focus."
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

only_product_img = remove(product_img)
def autocrop_image(img):
    bbox = img.getbbox()
    if bbox:
        cropped_img = img.crop(bbox)
        return cropped_img
msk = get_mask(only_product_img)
NOBG = autocrop_image(only_product_img)
processed_mask = mask_processor.preprocess(
    [msk], 
    height=1024, 
    width=1024
)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,     
    ip_adapter_image=NOBG,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=30,
    guidance_scale=7.5,
    cross_attention_kwargs={"ip_adapter_masks": [processed_mask]}
).images[0]
image.save("final_blue_banner.png")
display(image)

In [ ]:
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
product_img = load_image(url).convert("RGB").resize((1024, 1024))

image_np = np.array(product_img)
edges = cv2.Canny(image_np, 150, 300)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

pipe.set_ip_adapter_scale(0.9)

prompt = "Professional product photography of a can of soda standing on a smooth wet sea stone, turquoise ocean waves with white foam in the background, bright midday tropical sun, caustic water reflections, splash of sea water, high-end commercial aesthetic, 8k, highly detailed, sharp focus, vibrant colors."
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

only_product_img = remove(product_img)
def autocrop_image(img):
    bbox = img.getbbox()
    if bbox:
        cropped_img = img.crop(bbox)
        return cropped_img
msk = get_mask(only_product_img)
#NOBG = autocrop_image(only_product_img)
processed_mask = mask_processor.preprocess(
    [msk], 
    height=1024, 
    width=1024
)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,     
    ip_adapter_image=only_product_img,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=30,
    guidance_scale=7.5,
    cross_attention_kwargs={"ip_adapter_masks": [processed_mask]}
).images[0]
image.save("final_blue_banner.png")
display(image)

In [ ]:
def display_quad_collage(img_list, output_path="quad_collage.png", padding=20):
    """
    Creates and displays a horizontal collage of 4 square images.
    
    img_list: List containing 4 PIL Image objects.
    output_path: Filename to save the result.
    padding: Horizontal space (in pixels) between images.
    """
    
    if len(img_list) != 4:
        raise ValueError("This function requires exactly 4 images in the list.")

    # 1. Get dimensions from the first image (assuming all are the same size, e.g., 1024x1024)
    # We convert to RGB to ensure compatibility.
    imgs = [img.convert("RGB") for img in img_list]
    w, h = imgs[0].size

    # 2. Calculate final collage dimensions
    # Width: (4 * image_width) + (3 gaps * padding)
    collage_width = (4 * w) + (3 * padding)
    # Height: The height of a single image (assuming all are equal)
    collage_height = h

    # 3. Create the new empty canvas with a white background
    collage = Image.new('RGB', (collage_width, collage_height), color=(255, 255, 255))

    # 4. Paste images sequentially with horizontal offset
    current_x = 0
    for i in range(4):
        collage.paste(imgs[i], (current_x, 0))
        # Move the starting point for the next image by width + padding
        current_x += w + padding

    # 5. Save the result
    collage.save(output_path)
    print(f"Collage saved as {output_path}")

    return collage

def display_vertical_collage(img_list, output_path="vertical_collage.png", padding=40):
    if not img_list:
        print("Список изображений пуст.")
        return None

    imgs = [img.convert("RGB") for img in img_list]
    
    max_w = max(img.width for img in imgs)
    
    total_h = sum(img.height for img in imgs) + (padding * (len(imgs) - 1))

    collage = Image.new('RGB', (max_w, total_h), color=(255, 255, 255))

    current_y = 0
    for img in imgs:
        x_offset = (max_w - img.width) // 2
        collage.paste(img, (x_offset, current_y))
        current_y += img.height + padding

    collage.save(output_path)
    print(f"Вертикальный коллаж сохранен: {output_path}")
    IPython.display.display(collage)
    
    return collage

In [ ]:

def generate_image(prompt, ip_adapter_scale, url):
    product_img = load_image(url).convert("RGB").resize((1024, 1024))

    image_np = np.array(product_img)
    edges = cv2.Canny(image_np, 150, 300)
    edges = edges[:, :, None]
    edges = np.concatenate([edges, edges, edges], axis=2)
    canny_image = Image.fromarray(edges)
    pipe.set_ip_adapter_scale(ip_adapter_scale)

    negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

    only_product_img = remove(product_img)
    def autocrop_image(img):
        bbox = img.getbbox()
        if bbox:
            cropped_img = img.crop(bbox)
            return cropped_img
    msk = get_mask(only_product_img)
    #NOBG = autocrop_image(only_product_img)
    processed_mask = mask_processor.preprocess(
        [msk], 
        height=1024, 
        width=1024
    )
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=canny_image,     
        ip_adapter_image=only_product_img,
        controlnet_conditioning_scale=0.8,
        num_inference_steps=30,
        guidance_scale=7.5,
        cross_attention_kwargs={"ip_adapter_masks": [processed_mask]}
    ).images[0]
    image.save("final_blue_banner.png")
    return image

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

def create_labeled_collage(img_list, labels, output_path="labeled_collage.png", padding=20, label_height=80):
    """
    img_list: список из 4 PIL изображений
    labels: список из 4 строк (например, ["GS: 7.5", "GS: 10", ...])
    label_height: сколько пикселей добавить снизу для текста
    """
    if len(img_list) != len(labels):
        print("Ошибка: количество картинок и подписей должно совпадать!")
        return None

    font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    font_size = 70
    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()
        print("Шрифт CMU не найден, использую стандартный.")

    w, h = img_list[0].size
    collage_width = (w * len(img_list)) + (padding * (len(img_list) - 1))
    collage_height = h + label_height # Добавляем место под текст

    collage = Image.new('RGB', (collage_width, collage_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(collage)

    for i, (img, text) in enumerate(zip(img_list, labels)):
        x_offset = i * (w + padding)
        
        collage.paste(img.convert("RGB"), (x_offset, 0))
        
        text_bbox = draw.textbbox((0, 0), text, font=font)
        text_w = text_bbox[2] - text_bbox[0]
        
        text_x = x_offset + (w - text_w) // 2
        text_y = h + (label_height - font_size) // 2 # Центрируем в нижней полосе
        
        draw.text((text_x, text_y), text, font=font, fill=(0, 0, 0))

    collage.save(output_path)
    print(f"Коллаж с подписями сохранен: {output_path}")
    return collage



In [ ]:
prompt = "Professional product photography of a can of soda standing on a smooth wet sea stone, turquoise ocean waves with white foam in the background, bright midday tropical sun, caustic water reflections, splash of sea water, high-end commercial aesthetic, 8k, highly detailed, sharp focus, vibrant colors."
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"

ip_adapter_scales = [0.1, 0.3, 0.5, 0.7, 0.9]
imgs = []
for i in ip_adapter_scales:
    imgs.append(generate_image(prompt, i, url))
create_labeled_collage(imgs, list(map(str, ip_adapter_scales)))

In [ ]:
from PIL import Image
import IPython.display

def display_horizontal_collage(img_list, output_path="horizontal_collage.png", padding=20):
    """
    Создает и отображает горизонтальный коллаж из любого количества изображений.
    """
    if not img_list:
        print("Список изображений пуст.")
        return None

    # Конвертируем все в RGB
    imgs = [img.convert("RGB") for img in img_list]
    num_imgs = len(imgs)
    
    # Берем высоту по самому высокому, ширину суммируем
    max_h = max(img.height for img in imgs)
    total_w = sum(img.width for img in imgs) + (padding * (num_imgs - 1))

    # Создаем холст
    collage = Image.new('RGB', (total_w, max_h), color=(255, 255, 255))

    # Клеим по горизонтали
    current_x = 0
    for img in imgs:
        # Центрируем по вертикали, если картинки разной высоты
        y_offset = (max_h - img.height) // 2
        collage.paste(img, (current_x, y_offset))
        current_x += img.width + padding

    collage.save(output_path)
    print(f"Горизонтальный коллаж ({num_imgs} шт.) сохранен: {output_path}")
    
    return collage

In [ ]:
prompt = "Minimalist high-end product photography, [PRODUCT NAME] centered on a matte white surface, soft cinematic studio lighting, floating water droplets, sharp focus, neutral colors, professional advertising aesthetic, 8k, elegant composition."
url = "https://images.unsplash.com/photo-1605548230624-8d2d0419c517?ixlib=rb-4.1.0&q=85&fm=jpg&crop=entropy&cs=srgb&dl=deepal-tamang-D8XcEUt3Tmc-unsplash.jpg"
url1 = "https://www.hairbeautyink.com.au/cdn/shop/products/blonde-studio-post-lightening-shampoo-500ml-941238.jpg"
url2 = "https://extrabutterny.com/cdn/shop/files/MSRCXST-1.jpg?v=1688674811&width=1200"
Minimalist_img = []
for i in [url, url1, url2]:
    Minimalist_img.append(generate_image(prompt, 0.7, i))


In [ ]:
eco_friendly = display_horizontal_collage(Minimalist_img)
display(eco_friendly)

In [ ]:
vintage = display_horizontal_collage(Minimalist_img)
display(vintage)

In [ ]:
minimal = display_horizontal_collage(Minimalist_img)
display(minimal)


In [ ]:
display_vertical_collage([minimal, vintage, eco_friendly])

In [ ]:
Minimal